# 12. Reanotación ciega y segunda revisión

Este cuaderno crea dos hojas sin sugerencias automáticas ni etiquetas anteriores. La primera permite revisar 150 publicaciones después de un intervalo; la segunda permite que otra persona revise los 100 casos de prueba.

## A. Crear la muestra de 150

- **Qué hace:** sortea 150 IDs válidos con semilla 2026 y crea una hoja con lista desplegable.
- **Para qué:** medir la estabilidad de la misma persona al clasificar nuevamente.
- **Por qué:** la primera clasificación fue asistida por una sugerencia automática.

In [ ]:
%pip install -q openpyxl scikit-learn

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json

import pandas as pd
from IPython.display import display
from openpyxl import load_workbook
from openpyxl.worksheet.datavalidation import DataValidation
from sklearn.metrics import cohen_kappa_score, confusion_matrix

if Path('/content').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ = Path('/content/drive/MyDrive/TESIS_DATA_SCIENCE/COLAB')
else:
    RAIZ = Path.cwd()

CLASIFICACION = RAIZ / 'TABLAS/06_clasificacion/clasificacion_asistida_v5_etiquetada.xlsx'
PRUEBA = RAIZ / 'TABLAS/05_preparacion/muestra_prueba_100_v4.xlsx'
SALIDA = RAIZ / 'TABLAS/12_reanotacion'
SALIDA.mkdir(parents=True, exist_ok=True)

HOJA_150 = SALIDA / 'reanotacion_ciega_150.xlsx'
MANIFIESTO_150 = SALIDA / 'manifiesto_reanotacion_150.json'
HOJA_100 = SALIDA / 'segunda_persona_prueba_100.xlsx'
SEMILLA = 2026
N_REANOTACION = 150
DIAS_MINIMOS = 7
CATEGORIAS = ['positiva', 'negativa', 'neutral', 'inválida']

anotadas = pd.read_excel(CLASIFICACION, sheet_name='clasificacion', dtype=object, keep_default_na=False)
anotadas['id'] = pd.to_numeric(anotadas['id'], errors='raise').astype(int)
anotadas['etiqueta_humana'] = anotadas['etiqueta_humana'].astype(str).str.strip().str.lower()
validas = anotadas[anotadas['etiqueta_humana'].isin(CATEGORIAS[:3])].copy()

def agregar_lista(ruta, hoja, columna, fila_final):
    libro = load_workbook(ruta)
    ws = libro[hoja]
    validacion = DataValidation(
        type='list', formula1='"positiva,negativa,neutral,inválida"', allow_blank=True
    )
    ws.add_data_validation(validacion)
    validacion.add(f'{columna}2:{columna}{fila_final}')
    ws.freeze_panes = 'A2'
    libro.save(ruta)

if not HOJA_150.exists():
    muestra = validas[['id', 'texto_procesado']].sample(
        n=min(N_REANOTACION, len(validas)), random_state=SEMILLA
    ).rename(columns={'texto_procesado': 'texto'})
    muestra['etiqueta_reanotacion'] = ''
    muestra.to_excel(HOJA_150, sheet_name='reanotacion', index=False)
    agregar_lista(HOJA_150, 'reanotacion', 'C', len(muestra) + 1)
    MANIFIESTO_150.write_text(json.dumps({
        'fecha_seleccion_utc': datetime.now(timezone.utc).isoformat(),
        'semilla': SEMILLA,
        'n': int(len(muestra)),
        'dias_minimos': DIAS_MINIMOS,
    }, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Hoja creada:', HOJA_150.name)
display(pd.read_excel(HOJA_150).head(10))

## B. Calcular consistencia intraanotador

- **Qué hace:** compara la primera etiqueta con la segunda cuando la hoja está completa y se cumple el intervalo.
- **Para qué:** obtener acuerdo observado, kappa y tabla cruzada.
- **Por qué:** permite observar si la segunda revisión mueve más casos hacia neutral.

In [ ]:
segunda = pd.read_excel(HOJA_150, sheet_name='reanotacion', dtype=object, keep_default_na=False)
segunda['etiqueta_reanotacion'] = segunda['etiqueta_reanotacion'].astype(str).str.strip().str.lower()
faltan_150 = int((~segunda['etiqueta_reanotacion'].isin(CATEGORIAS)).sum())
if faltan_150:
    print(f'[COMPLETAR] Faltan {faltan_150} etiquetas en la hoja de 150.')
else:
    meta = json.loads(MANIFIESTO_150.read_text(encoding='utf-8'))
    inicio = datetime.fromisoformat(meta['fecha_seleccion_utc'])
    dias = (datetime.now(timezone.utc) - inicio).total_seconds() / 86400
    if dias < DIAS_MINIMOS:
        print(f'[COMPLETAR] Han transcurrido {dias:.2f} días; el mínimo configurado es {DIAS_MINIMOS}.')
    else:
        pares = segunda[['id', 'etiqueta_reanotacion']].merge(
            anotadas[['id', 'etiqueta_humana']], on='id', validate='one_to_one'
        )
        acuerdo = float((pares['etiqueta_humana'] == pares['etiqueta_reanotacion']).mean())
        kappa = float(cohen_kappa_score(
            pares['etiqueta_humana'], pares['etiqueta_reanotacion'], labels=CATEGORIAS
        ))
        cruzada = pd.crosstab(
            pares['etiqueta_humana'], pares['etiqueta_reanotacion'], dropna=False
        ).reindex(index=CATEGORIAS, columns=CATEGORIAS, fill_value=0)
        resultado = {
            'n': int(len(pares)), 'dias_transcurridos': dias,
            'acuerdo_observado': acuerdo, 'kappa_intraanotador': kappa,
            'neutral_primera': int((pares.etiqueta_humana == 'neutral').sum()),
            'neutral_segunda': int((pares.etiqueta_reanotacion == 'neutral').sum()),
        }
        (SALIDA / 'resultado_reanotacion_150.json').write_text(
            json.dumps(resultado, ensure_ascii=False, indent=2), encoding='utf-8'
        )
        cruzada.to_excel(SALIDA / 'tabla_cruzada_reanotacion_150.xlsx')
        print(resultado)
        display(cruzada)

## C. Preparar la revisión de los 100 casos por otra persona

- **Qué hace:** crea una hoja ciega de la prueba y calcula kappa cuando se completa.
- **Para qué:** estimar acuerdo entre personas sobre la referencia de evaluación.
- **Por qué:** una segunda revisión independiente fortalece la interpretación de la prueba.

In [ ]:
prueba = pd.read_excel(PRUEBA, sheet_name='evaluacion', dtype=object, keep_default_na=False)
prueba['id'] = pd.to_numeric(prueba['id'], errors='raise').astype(int)
prueba['etiqueta_polaridad'] = prueba['etiqueta_polaridad'].astype(str).str.strip().str.lower()

if not HOJA_100.exists():
    hoja = prueba[['id', 'texto_procesado']].rename(columns={'texto_procesado': 'texto'}).copy()
    hoja['etiqueta_revisor_2'] = ''
    hoja.to_excel(HOJA_100, sheet_name='revision', index=False)
    agregar_lista(HOJA_100, 'revision', 'C', len(hoja) + 1)
    print('Hoja creada:', HOJA_100.name)

revision = pd.read_excel(HOJA_100, sheet_name='revision', dtype=object, keep_default_na=False)
revision['etiqueta_revisor_2'] = revision['etiqueta_revisor_2'].astype(str).str.strip().str.lower()
faltan_100 = int((~revision['etiqueta_revisor_2'].isin(CATEGORIAS)).sum())
if faltan_100:
    print(f'[COMPLETAR] Faltan {faltan_100} etiquetas de la segunda persona.')
else:
    pares = revision[['id', 'etiqueta_revisor_2']].merge(
        prueba[['id', 'etiqueta_polaridad']], on='id', validate='one_to_one'
    )
    acuerdo = float((pares.etiqueta_revisor_2 == pares.etiqueta_polaridad).mean())
    kappa = float(cohen_kappa_score(
        pares.etiqueta_polaridad, pares.etiqueta_revisor_2, labels=CATEGORIAS
    ))
    matriz = pd.DataFrame(
        confusion_matrix(
            pares.etiqueta_polaridad, pares.etiqueta_revisor_2, labels=CATEGORIAS
        ), index=CATEGORIAS, columns=CATEGORIAS
    )
    resultado = {'n': int(len(pares)), 'acuerdo_observado': acuerdo, 'kappa_interanotador': kappa}
    (SALIDA / 'resultado_segunda_persona.json').write_text(
        json.dumps(resultado, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    matriz.to_excel(SALIDA / 'matriz_segunda_persona.xlsx')
    print(resultado)
    display(matriz)
display(revision.head(10))